In [1]:
import pandas as pd
import numpy as np

# 1. Load data (Set dtype to ensure identifiers like CIK/SIC retain leading zeros)
df = pd.read_csv("sec_risk_factors.csv", dtype={"cik": str, "sic": str})

# 2. Verify basic dataset coverage
print(f"Total rows (Filings): {len(df)}")
print(f"Companies included: {df['ticker'].unique().tolist()}")
print(f"Filing years range: {df['filing_year'].min()} - {df['filing_year'].max()}")

# 3. Check for missing values and inspect word count distribution
print("\n--- Missing Values Check ---")
print(df[["ticker", "filing_date", "risk_text", "word_count"]].isnull().sum())

print("\n--- Word Count Summary Statistics by Ticker ---")
print(df.groupby("ticker")["word_count"].describe())

Total rows (Filings): 15
Companies included: ['ADBE', 'MSFT', 'ORCL']
Filing years range: 2021 - 2025

--- Missing Values Check ---
ticker         0
filing_date    0
risk_text      0
word_count     0
dtype: int64

--- Word Count Summary Statistics by Ticker ---
        count     mean          std      min      25%      50%      75%  \
ticker                                                                    
ADBE      5.0  10823.2  1037.071454   9722.0   9801.0  10951.0  11685.0   
MSFT      5.0  10323.4   327.296960  10104.0  10119.0  10143.0  10377.0   
ORCL      5.0  11229.2   396.529570  10683.0  11052.0  11189.0  11556.0   

            max  
ticker           
ADBE    11957.0  
MSFT    10874.0  
ORCL    11666.0  


In [2]:
# 1. 定义网络安全关键词列表
keywords = ['cybersecurity', 'cyber', 'breach', 'security', 'incident', 'unauthorized']

# 2. 计算文本中关键词出现的总频次 (忽略大小写)
df['cyber_kw_count'] = df['risk_text'].apply(
    lambda text: sum(text.lower().count(kw) for kw in keywords) if isinstance(text, str) else 0
)

# 3. 计算标准化指标：每 10,000 字中的关键词频率 (Standardized Frequency per 10k words)
df['kw_density_per_10k'] = (df['cyber_kw_count'] / df['word_count']) * 10000

# 4. 查看各公司历年的网络安全词汇密度趋势
pivot_trend = df.pivot(index='filing_year', columns='ticker', values='kw_density_per_10k')
print("\n--- 各公司历年网络安全词汇密度 (Per 10,000 words) ---")
print(pivot_trend.round(2))


--- 各公司历年网络安全词汇密度 (Per 10,000 words) ---
ticker         ADBE    MSFT   ORCL
filing_year                       
2021          53.92   65.07  43.06
2022          57.71   70.35  42.90
2023          64.83   76.21  47.05
2024         109.03   90.12  45.00
2025         105.09  101.79  43.72


In [3]:
import pandas as pd
import re

df = pd.read_csv("sec_risk_factors.csv", dtype={"cik": str, "sic": str})

# 定义新规涉及的三大维度关键词
topic_kws = {
    "Governance (治理/监督)": ["board", "committee", "oversight", "governance", "management"],
    "Incidents (事件/攻击)": ["incident", "breach", "attack", "ransomware", "unauthorized"],
    "Third-Party (第三方/供应链)": ["vendor", "supplier", "third party", "third-party", "supply chain", "cloud"]
}

# 计算三大维度的每万字密度
for topic, terms in topic_kws.items():
    pattern = r'\b(' + '|'.join(terms) + r')\b'
    df[topic] = df['risk_text'].apply(lambda x: len(re.findall(pattern, str(x), re.IGNORECASE)))
    df[f"{topic} (Density)"] = (df[topic] / df['word_count']) * 10000

# 提取核心数据进行查看
trend = df.pivot(index='filing_year', columns='ticker', values=[f"{t} (Density)" for t in topic_kws.keys()])
print("--- 细分维度密度变化趋势 ---")
print(trend.round(1))

--- 细分维度密度变化趋势 ---
            Governance (治理/监督) (Density)             \
ticker                              ADBE MSFT  ORCL   
filing_year                                           
2021                                12.0  7.9  15.0   
2022                                11.7  7.7  14.3   
2023                                12.8  8.9  12.7   
2024                                18.5  9.2  14.7   
2025                                22.4  8.9  12.9   

            Incidents (事件/攻击) (Density)              \
ticker                             ADBE  MSFT  ORCL   
filing_year                                           
2021                               22.3  14.8  12.2   
2022                               20.9  15.4  10.7   
2023                               22.8  16.8  10.9   
2024                               25.7  24.8  10.4   
2025                               24.5  25.7   9.4   

            Third-Party (第三方/供应链) (Density)              
ticker                                 AD

In [5]:
# 提取 Adobe (ADBE) 2023 与 2024 年新增的网络安全关键句
adbe_23_text = df[(df['ticker'] == 'ADBE') & (df['filing_year'] == 2023)]['risk_text'].values[0]
adbe_24_text = df[(df['ticker'] == 'ADBE') & (df['filing_year'] == 2024)]['risk_text'].values[0]

# 简易分句并过滤包含 security / cyber 的句子
sents_23 = set([s.strip() for s in adbe_23_text.split('.') if 'cyber' in s.lower() or 'security' in s.lower()])
sents_24 = set([s.strip() for s in adbe_24_text.split('.') if 'cyber' in s.lower() or 'security' in s.lower()])

# 寻找 2024 年出现但 2023 年未包含的前缀文本
new_sentences = [s for s in sents_24 if not any(s[:30] in s2 for s2 in sents_23)]

print(f"\n--- ADBE 2024 年新增的网络安全句子片段 (共 {len(new_sentences)} 条) ---")
for i, sent in enumerate(new_sentences[:3], 1):
    print(f"[{i}] {sent[:180]}...\n")


--- ADBE 2024 年新增的网络安全句子片段 (共 23 条) ---
[1] There are numerous ways that our reputation or brands could be damaged, including, among other things, introduction of new products, features or services that do not meet customer ...

[2] These risks and uncertainties include: • inability to achieve the financial and strategic goals of the investment or acquisition; • difficulty in effectively integrating the operat...

[3] Actual or perceived security vulnerabilities or incidents may result in claims or litigation and liability or fines (and have in the past led to such claims), costly and time-inten...



In [6]:
import pandas as pd

df = pd.read_csv("sec_risk_factors.csv")

# 计算相比前一年的字数增减量
df['word_change'] = df.groupby('ticker')['word_count'].diff()

# 查看每年的字数变化表
word_trends = df.pivot(index='filing_year', columns='ticker', values='word_change')
print("--- 每年风险章节字数净增减 (Word Count Deltas) ---")
print(word_trends)

--- 每年风险章节字数净增减 (Word Count Deltas) ---
ticker         ADBE   MSFT   ORCL
filing_year                      
2021            NaN    NaN    NaN
2022          272.0  234.0  506.0
2023        -1006.0 -273.0 -137.0
2024        -1229.0  770.0  504.0
2025           79.0 -755.0  110.0


In [7]:
import re
from collections import Counter

# 过滤无意义停用词
stopwords = set(['the', 'and', 'to', 'of', 'a', 'in', 'or', 'our', 'is', 'for', 'that', 'by', 'as', 'with', 'on', 'be', 'may', 'are', 'from', 'at', 'this', 'an', 'not', 'we', 'such', 'other', 'which', 'have', 'all', 'new', 'including', 'their', 'has', 'more', 'if', 'any', 'also', 'it', 'can', 'business', 'these', 'could', 'would', 'results', 'financial'])

print("--- 各公司最常出现的高频焦点词 (Top 8 Words) ---")
for ticker in df['ticker'].unique():
    text = " ".join(df[df['ticker'] == ticker]['risk_text']).lower()
    words = [w for w in re.findall(r'\b[a-z]{4,}\b', text) if w not in stopwords]
    top_words = [word for word, count in Counter(words).most_common(8)]
    print(f"{ticker}: {', '.join(top_words)}")

--- 各公司最常出现的高频焦点词 (Top 8 Words) ---
ADBE: customers, products, services, data, third, future, operations, impact
MSFT: services, products, customers, data, security, operations, software, result
ORCL: products, services, cloud, customers, operations, affect, offerings, hardware
